# TD Methods: SARSA and Q-Learning

In [ ]:
!pip install gymnasium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 958.1/958.1 kB 9.4 MB/s eta 0:00:00


In [ ]:
# Video management imports
import cv2
import matplotlib.pyplot as plt

# Check if we running in Google Colab or Jupyter Notebook
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print('Running in Google Colab')
    # Do you need to connect with Google Drive? Uncomment the next two lines
    # from google.colab import drive
    # drive.mount('/content/drive')
    # This auxiliary function simplifies the visualization of OpenCV Images
    from google.colab.patches import cv2_imshow
else:
    print('Running in Jupyter Notebook')
    # This auxiliary function simplifies the visualization of OpenCV Images
    def cv2_imshow(img, title=''):
        if img.ndim > 2:
            img= cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            # view the image in its natural size
            plt.figure(figsize=(img.shape[1], img.shape[0]), dpi=1)
            plt.imshow(img)
            plt.title(title)
            plt.xticks([]), plt.yticks([])
            plt.show()
        else:
            # view the image in its natural size
            plt.figure(figsize=(img.shape[1], img.shape[0]), dpi=1)
            plt.imshow(img, cmap='gray')
            plt.title(title)
            plt.xticks([]), plt.yticks([])
            plt.show()

# Helper functions to save videos and images
def save_video(img_array, path='/content/test.mp4'):
  height, width, layers = img_array[0].shape
  size = (width, height)
  out = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*'MP4V'), 15, size)
  if out.isOpened():
    for i in range(len(img_array)):
      bgr_img = cv2.cvtColor(img_array[i], cv2.COLOR_RGB2BGR)
      out.write(bgr_img)
    out.release()
    print('Video saved.')
  else:
    print(f'Could not save video path: {path}')

Running in Google Colab


## The Environment: Cliff Walking
(From R. S. Sutton and A. G. Barto, Reinforcement Learning: An Introduction, 2nd ed. Cambridge, MA: The MIT Press, 2018)

![Cliff Walking Grid World](https://miro.medium.com/max/700/1*52MwrYKyzQXuKZ88rqu70A.png)

This is a standard undiscounted, episodic task, with start and goal states, and the usual actions causing movement up, down, right, and left. Reward is -1 on all transitions except those into the region marked "The Cliff". Stepping into this region incurs a reward of -100 and sends the agent instantly back to the start.

In this environment, the position of a cell is encoded in the following way:
```
|(0, 0) (0, 1) ... (0, 11)|
|(1, 0) (1, 1) ... (1, 11)|
|(2, 0) (2, 1) ... (2, 11)|
|(3, 0) (3, 1) ... (3, 11)|
```
If we number the cells with consecutively, row-wise, we have:
```
| 0  1 ... 11|
|12 13 ... 23|
|24 25 ... 35|
|36 37 ... 47|
```
Which is the encoding of the observations of the environment.

The observation space is just the number in [0, 47] of the cell where the agent is.

The action space is a value:
- 0: Up
- 1: Right
- 2: Down
- 3: Left

The agent always starts on cell 36 and has to reach the cell 47.

More information: The CliffWalking environment [Gymnasium - CliffWalking](https://gymnasium.farama.org/environments/toy_text/cliff_walking/)

In [ ]:
import gymnasium as gym
import numpy as np
import plotly.express as px
from tqdm.notebook import tqdm

print(f"Gym version: {gym.__version__}")

Gym version: 1.0.0


## Base TD Agent

First, we will define a helper function to implement argmax handling the case where there are several values with the maximum value (a tie). In this case, the function will select randomly among them. Thus, we avoid overrepresenting certain actions.

In [ ]:
# Helper function representing the argmax function. This implementation
# finds all the elements matching the maximun value and selects one,
# randomly, to break the tie.
def argmax(np_array):
    """argmax method with random tie-breaking.

    Args:
      np_array: A numpy array.
    Returns:
      Index of one of the appearances of the highest value in the array.
    """
    tie_indices = np.flatnonzero(np_array == np_array.max())
    return np.random.choice(tie_indices)

For this example, we will define a base class for both methods containing some basic functions, such as choose_policy_action and choose_e_greedy_action. The implementation is exactly the same as in the Montecarlo methods.

In [ ]:
class TDAgent:
  def __init__(self, /, num_states, num_actions, gamma = 1.0, alpha = 0.5, epsilon = 0.1):
    # Future discount parameter
    self.gamma = gamma
    # Learning rate
    self.alpha = alpha
    # Epsilon parameter for the e-greedy policy
    self.epsilon = epsilon
    # Number of states
    self.num_states = num_states
    # Number of actions
    self.num_actions = num_actions
    # Q-values
    self.q_values = np.zeros((self.num_states, self.num_actions))

  # Choose the action according to the currently learned policy, i.e. selecting
  # the action that maximized the value.
  def choose_policy_action(self, observation):
    return argmax(self.q_values[observation, :])

  # Choose the action following an epsilon-greedy policy
  def choose_e_greedy_action(self, observation):
      # We select with a given probability a random action
      if np.random.rand() < self.epsilon:
          return np.random.choice(range(self.num_actions))
      # or otherwise we follow the policy action.
      else:
          return self.choose_policy_action(observation)

## SARSA Agent

SARSA updates the action-value function Q at each step. The update rule for SARSA is:

$$ Q(S_t, A_t) = Q(S_t, A_t) + \alpha(R_{t+1} + \gamma Q(S_{t+1}, A_{t+1}) - Q(S_t, A_t)) $$

Notice that the next observation and action are chosen before the update under the current policy $\pi$.

In [ ]:
class SarsaAgent(TDAgent):
  # Updates the agent's Q-Values.
  #  The Q-Values are updated using the bellman update
  #  Q(S_t, A_t) = Q(S_t, A_t) + alpha * [R_t+1 + gamma * Q(S_t+1, A_t+1) - Q(S_t, A_t)]
  def update_q(self, observation, action, next_observation, next_action, reward, done):
    # Calculate the SARSA update
    # Which is the value of the TD target for the final state?
    td_target = reward
    if(not done):
      # Which is the value of the TD target for non-ending states?
      # Use self.q_values and the discounting parameter self.gamma
      td_target += self.gamma * self.q_values[next_observation, next_action]
    # The SARSA update using the calculated td_target variable?
    # Hint: You have the current state and action stored in
    # self.observation and self.action. The learning rate is self.alpha.
    self.q_values[observation, action] += self.alpha * (td_target - self.q_values[observation, action])

### Training

Notice how the training loop changes from the case of Montecarlo methods. Here we perform the update Q at each step. To compute it, we use the observations and rewards obtained for the next step.

In [ ]:
# Num of episodes to learn
EPISODES = 5000
# Initialize the environment
env = gym.make('CliffWalking-v0', render_mode='rgb_array')

# Define agent
sarsa_agent = SarsaAgent(num_states=(4 * 12), num_actions= 4, gamma= 1.0, alpha= 0.6, epsilon= 0.1)
# Tracking the training
visit_count = np.zeros(4*12)

for episode in tqdm(range(EPISODES)):
    done = False
    # Get first observation
    observation, _ = env.reset()
    visit_count[observation] += 1
    # Choose an initial action
    action = sarsa_agent.choose_e_greedy_action(observation)
    while not done:
        # Perform the step with currently selected action
        next_observation, reward, terminated, truncated, _ = env.step(action)
        # Check if we are done
        done = terminated or truncated

        # We have to choose the next action
        next_action = sarsa_agent.choose_e_greedy_action(next_observation)

        # Have the agent update its Q-Values
        sarsa_agent.update_q(observation, action, next_observation, next_action, reward, done)

        # Update current observation and action to the new ones
        observation = next_observation
        action = next_action

        # Update new observation count
        visit_count[observation] += 1

  0%|          | 0/5000 [00:00<?, ?it/s]

### Training Evolution

In [ ]:
fig = px.imshow(visit_count.reshape((4, 12)))
fig.show()

In [ ]:
print(sarsa_agent.q_values)

[[ -23.8442715   -18.69211854  -31.07088983  -30.43727142]
 [ -24.87533887  -14.62478871  -28.55916583  -28.61458273]
 [ -25.58939892  -13.17575001  -21.83859104  -22.27376508]
 [ -24.2815973   -11.9779955   -26.34646362  -27.01642525]
 [ -19.85517406  -10.88120205  -20.11724705  -22.0276129 ]
 [ -17.0439982    -9.63915726  -17.24491171  -17.98942406]
 [ -13.59404498   -8.55849017  -23.87208769  -20.58492998]
 [ -12.35888102   -7.59488588  -14.47404099  -13.29294683]
 [ -13.39960905   -6.35274645  -12.92493857  -14.2727935 ]
 [  -8.11122527   -5.07648917  -37.16139191  -12.76906348]
 [  -5.39525653   -4.00002859  -21.78749968   -8.22619178]
 [  -5.22561353   -6.28180424   -3.00000253   -7.5800385 ]
 [ -26.73643011  -18.11831687  -29.83260671  -30.46321417]
 [ -16.30552327  -35.56089435  -29.19927708  -30.14364875]
 [ -21.91821208  -31.94154585  -28.18896504  -31.00451381]
 [ -23.7555672   -26.85318943  -30.3834035   -24.03539556]
 [ -16.11165394  -26.52217058  -26.14922659  -27.1990097

### Testing

In [ ]:
observation, _ = env.reset()
done = False
images = []
while not done:
    action = sarsa_agent.choose_policy_action(observation)
    observation, reward, terminated, truncated, _ = env.step(action)
    done = terminated or truncated
    image = env.render()
    images.append(image)

    if len(images) > 100:
        done = True

save_video(images, path='/content/SARSA_cliffhanger.mp4')

Video saved.


## Q-Learning Agent

Q-Learning updates the action-value function Q at each step, as SARSA. The update rule for Q-Learning is:

$$ Q(S_t, A_t) = Q(S_t, A_t) + \alpha(R_{t+1} + \gamma \max_{a} Q(S_{t+1}, a) - Q(S_t, A_t)) $$

In this case, the action used to compute the target is different from the one used for the behavior of the agent.

Notice how we use the max action-value possible for the next observation. Thus, we estimate the value of the state considering the best possible value even if the actual action taken is different.

In [ ]:
class QLearningAgent(TDAgent):
  #Updates the agent's Q-Values.
  # The Q-Values are updated using Bellman's update:
  # Q(S_t, A_t) = Q(S_t, A_t) + alpha * [R_t+1 + gamma * max_a(Q(S_t+1, a)) - Q(S_t, A_t)]
  def update_q(self, observation, action, next_observation, reward, done):
    # Calculate the Q-Learning update.
    # Which is the value of the TD target for the final state?
    td_target = reward
    if not done:
        # Which is the value of the TD target for non-ending states?
        # In this case we use self.q_values, the discounting parameter self.gamma, and
        # np.max to calculate the maximum Q-value.
        td_target += self.gamma * np.max(self.q_values[next_observation, :])

    td_error = (td_target - self.q_values[observation, action])
    self.q_values[observation, action] += self.alpha * td_error


### Training

In Q-learning we follow a standard simulation loop:
1. choose action,
2. perform step,
3. update Q,
as don't need the actual observations of the next

In [ ]:
# Num of episodes to learn
EPISODES = 5000
# Initialize the environment
env = gym.make('CliffWalking-v0', render_mode ='rgb_array')

# Define agent
q_agent = QLearningAgent(num_states= (4 * 12), num_actions= 4, gamma= 1.0, alpha= 0.9, epsilon= 0.1)
# Tracking the training
visit_count = np.zeros(4*12)

for episode in tqdm(range(EPISODES)):
    done = False
    # Get first observation
    observation, _ = env.reset()
    visit_count[observation] += 1
    while not done:
        # Choose action
        action = q_agent.choose_e_greedy_action(observation)
        # Take a step in env
        next_observation, reward, done, _, _ = env.step(action)
        # Have the agent update its Q-Values
        q_agent.update_q(observation, action, next_observation, reward, done)
        # Update agent observation to new one
        observation = next_observation
        visit_count[observation] += 1

  0%|          | 0/5000 [00:00<?, ?it/s]

### Training Evolution

In [ ]:
fig = px.imshow(visit_count.reshape((4, 12)))
fig.show()

In [ ]:
print(q_agent.q_values)

[[ -12.771       -12.61911675  -13.85653289  -12.69      ]
 [ -12.89184013  -12.94995751  -12.57966721  -12.75524267]
 [ -12.95933776  -11.99952866  -11.99992642  -12.52879379]
 [ -11.871       -10.9999573   -10.99983952  -11.59402317]
 [ -10.9995014    -9.99998441   -9.99999971  -11.05799236]
 [  -9.42210046   -8.99999598   -8.99999902   -9.80182921]
 [  -8.80520149   -7.99999909   -7.99999999   -9.79762295]
 [  -7.94631851   -6.99999979   -6.99999993   -8.89408344]
 [  -6.3          -5.99999996   -6.           -7.98526676]
 [  -5.4          -4.99999999   -4.99999999   -6.33485074]
 [  -4.88481003   -4.           -4.           -4.491     ]
 [  -3.96         -3.9599995    -3.           -4.3371    ]
 [ -13.5321615   -13.          -13.          -14.        ]
 [ -12.98454983  -12.          -12.          -13.99807685]
 [ -12.99911767  -11.          -11.          -12.99999929]
 [ -11.9997327   -10.          -10.          -11.99999999]
 [ -10.99997003   -9.           -9.          -11.       

### Testing

In [ ]:
observation, _ = env.reset()
done = False
images = []
while not done:
    action = q_agent.choose_policy_action(observation)
    observation, reward, done, _, _ = env.step(action)

    image = env.render()
    images.append(image)

    if len(images)>100:
        done = True

save_video(images, path='/content/Q-Learning_cliffhanger.mp4')

Video saved.


# Appendix

## `np_array.max()`
Obtains the highest value on the array.

In [ ]:
test_array = np.array([0, 7, 3, 8, 4, 3, 9, 9, 2])
test_array.max()

9

## `np.flatnonzero(np_array)`

This method can be used to get the indices of the values in the array that are different from zero or zero-like (e.g. False).

In [ ]:
test_array = np.array([0, 7, 4, 0, 7, 5, 0, 3])
np.flatnonzero(test_array)

array([1, 2, 4, 5, 7])

In [ ]:
test_array = np.array([False, True, False, True, True, False])
np.flatnonzero(test_array)

array([1, 3, 4])

## `np.random.choice(np_array)`
Returns a random sample from the given array.

In [ ]:
test_array = np.array([4, 7, 11, 1, 8])
np.random.choice(test_array)

11

## `np.random.rand()`
Returns a random sample from 0 to 1.

In [ ]:
np.random.rand()

0.8176661208357554